<h2>Dates and calendars and conventions</h2>

specify an explicit first date; we’ll use the schedule to specify coupon dates for a bond, and this allows us to have a long first coupon from August 2020 to May 2021, followed by regular semiannual coupons resetting in November and May until 2031.

In [1]:
import QuantLib as ql

In [2]:
value_date = ql.Date(8, ql.December, 2022)
ql.Settings.instance().evaluationDate = value_date

schedule = ql.MakeSchedule(
    effectiveDate = ql.Date(26, ql.August, 2020),
    terminationDate = ql.Date(26, ql.May, 2031),
    firstDate=ql.Date(26, ql.May, 2021),
    frequency=ql.Semiannual,
    calendar=ql.TARGET(),
    convention=ql.Following,
    backwards=True
)

for d in schedule:
    print(d)

August 26th, 2020
May 26th, 2021
November 26th, 2021
May 26th, 2022
November 28th, 2022
May 26th, 2023
November 27th, 2023
May 27th, 2024
November 26th, 2024
May 26th, 2025
November 26th, 2025
May 26th, 2026
November 26th, 2026
May 26th, 2027
November 26th, 2027
May 26th, 2028
November 27th, 2028
May 28th, 2029
November 26th, 2029
May 27th, 2030
November 26th, 2030
May 26th, 2031


<h2>Some simple bond calculations</h2>

In [3]:
settlement_days = 3
face_amount = 1000000.0
coupons = [0.02]
day_counter = ql.Thirty360(ql.Thirty360.BondBasis)

bond = ql.FixedRateBond(
    settlement_days,
    face_amount,
    schedule,
    coupons,
    day_counter
)

bond_yield = 0.025
price = bond.cleanPrice(bond_yield, day_counter, ql.Compounded, ql.Annual)
accrual = bond.accruedAmount()

print(f"Price: {price:.6}")
print(f"Accrual: {accrual: .4}")

Price: 96.3256
Accrual:  0.08333


<h2>Discount curves</h2>
Whatever the curve, it can be used to calculate a theoretical price for the bond; instead of relying on a quoted yield, we can create a pricing engine that first uses the curve to discount each bond coupon to the settlement date and then accumulates the results. Once created, we pass the engine to the bond and ask it for its price.

In [4]:
curve_nodes = [
    value_date,
    value_date + ql.Period(1, ql.Years),
    value_date + ql.Period(2, ql.Years),
    value_date + ql.Period(5, ql.Years),
    value_date + ql.Period(10, ql.Years),
    value_date + ql.Period(15, ql.Years),
]
curve_rates = [
    0.015,
    0.015,
    0.018,
    0.022,
    0.025,
    0.028
]
#A zero curve is a line that shows the interest rates on zero-coupon bonds across different future maturity dates.
issuer_curve = ql.ZeroCurve(curve_nodes, curve_rates, ql.Actual360())

bond.setPricingEngine(
    ql.DiscountingBondEngine(ql.YieldTermStructureHandle(issuer_curve))
)
print(f"Price: {bond.cleanPrice():.6}")


Price: 96.6641


<h2>Pricing at a spread over a curve</h2>
The spread (modeled as an instance of the SimpleQuote class) is initially set to zero; but it can be set afterwards to any other value, and the bond price will react accordingly. Adding a constant spread is not the only way we can modify an existing curve: other classes provide the means to model, say, flattening or steepening scenarios.


In [6]:
curve_rates_2 = [
    -0.005,
    -0.005,
    0.001,
    0.004,
    0.009,
    0.012
]
govies_curve = ql.ZeroCurve(curve_nodes, curve_rates_2,ql.Actual360())

spread = ql.SimpleQuote(0.0)
discount_curve = ql.ZeroSpreadedTermStructure(
    ql.YieldTermStructureHandle(govies_curve), ql.QuoteHandle(spread)
)

bond.setPricingEngine(
    ql.DiscountingBondEngine(ql.YieldTermStructureHandle(discount_curve))
)

print(f"Price: {bond.cleanPrice():.6}")

spread.setValue(0.01)

print(f"Price : {bond.cleanPrice():.6}")


Price: 110.34
Price : 101.913


<h2>Finding the z-spread given a price</h2>
Finally, what if you have a price—for instance, the one we calculated given the quoted yield—and you want to find the corresponding z-spread over the government curve? 


In [ ]:
def discrepancy(s):
    spread.setValue(s)
    return bond.cleanPrice() - price


solver = ql.Brent()
accuracy = 1e-5
guess = 0.015
step = 1e-4

spread_over_govies = solver.solve(discrepency, accuracy, guess, step)

print(f"Spread: {spread_over_govies:}")